# How to Visualize Voltage Glitches

Our [fault101 notebooks](https://github.com/newaetech/chipwhisperer-jupyter/tree/main/courses/fault101) teach how to use voltage glitching to change a target's behaviour.

The change in target behaviour is "proof" that glitching works. However voltage glitching attacks are not deterministic and may not work right away without some additional effort on your part. The reason for this is that glitch attacks in general can be quite sensitive to environmental factors (voltage, temperature), variances in manufacturing processes, and even things like the length and characteristics of your SMA cable. Even after finding a successful combination of glitch parameters for your own setup, these glitches may not have the intended effect all of the time.

Whatever the reason, users sometimes wish to "see" the glitch on an oscilloscope. This note will teach you how.

Supported Capture Hardware:

✅ CW-Nano\
✅ CW-Lite\
✅ CW-Pro\
✅ CW-Husky

Required ChipWhisperer software installation:
✅ any release

## Target Required!

The first thing to understand is that **you won't see any glitches unless you have a target connected**.

If you connect an oscilloscope directly to your device's glitch port, you won't see anything interesting.

To explain why, let's first review how ChipWhisperer implements voltage glitching.

The figure below is from Colin's ["Fault Injection using Crowbars on Embedded Systems" paper](https://eprint.iacr.org/2016/810.pdf):

![](../Images/crowbar.png)

Pretty simple: driving the "input" line high turns on the MOSFET and connects Vcc to ground through a small shunt resistor.

This is very simple for a single target; to make this work in the ChipWhisperer hardware ecosystem, where different targets have different supply voltages, and different shunt resistors, we needed to partition things in a clever way:
- the MOSFET is in your capture device (e.g. CW-Husky)
- the shunt resistor in our your target board (e.g. your [SAM4S target board](../../chipwhisperer-target-cw308t/CW312T_ATSAM4S/README))
- most of the connections between the various components are on your target adapter board (e.g. your [CW313](../../Targets/CW313))

If you look over the schematics for these three pieces, you'll see how they build the logical equivalent of figure 1 above. It may seem unnecessarily complicated, but this is how different targets can be accommodated (without e.g. placing the MOSFET directly on every single target board that we make).

TL;DR: **don't connect your oscilloscope probe to the glitch port**; you won't see anything that way.

## Inferring Glitches from Power Traces

Let's start by looking at a voltage glitch on a normal power trace capture. This works the same on all ChipWhisperer hardware and doesn't require any external equipment, other than a ChipWhisperer target.

Set the following as per your target:

In [1]:
PLATFORM = 'CW308_SAM4S'
SS_VER = "SS_VER_1_1"

In [2]:
%run ../../../../jupyter/Setup_Scripts/Setup_Generic.ipynb

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 137209                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen            

We'll use our `simpleserial-trace` firmware, but `simpleserial-aes` compiled for any of our targets will also work (or anything you desire, really, if you change the `cw.capture_trace()` call below as needed).

In [3]:
cw.program_target(scope, prog, '../fw/simpleserial-trace-SSv1.hex')

In [4]:
reset_target(scope)

We first grab a basic power trace, with no glitches:

In [5]:
scope.trigger.module = 'basic'
scope.trigger.triggers = 'tio4'

scope.adc.samples = 32000
scope.adc.presamples = 0
scope.adc.segments = 1
scope.adc.bits_per_sample = 8  # SAD is done at 8 bits per sample

scope.gain.db = 22

scope.glitch.enabled = False

In [6]:
reftrace = cw.capture_trace(scope, target, bytearray(16), bytearray(16), as_int=False)

In [7]:
cw.plot(reftrace.trace)

:Curve   [x]   (y)

Now we enable glitching. To make the glitch more visually obvious, we'll make it two full target clock cycles long:

In [8]:
scope.glitch.enabled = True
scope.glitch.trigger_src = 'ext_single'
scope.glitch.clk_src = 'pll'
scope.glitch.output = 'enable_only'
scope.glitch.repeat = 2

scope.io.glitch_hp = True

We'll position the glitch 1000 cycles after the trigger, again to make it easier to find:

In [9]:
scope.glitch.ext_offset = 1000

In [10]:
glitchtrace = cw.capture_trace(scope, target, bytearray(16), bytearray(16), as_int=False)

Depending on your target, the glitch may have caused the ADC to clip:

In [11]:
scope.adc.errors

'ADC clipped, '

In [12]:
scope.errors.clear()

Now let's compare the glitched and non-glitched power traces:

In [14]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import Span
from bokeh.models import Span, Legend, LegendItem

output_notebook()

p = figure(width=800, tools='pan, box_zoom, hover, reset, save')

xrange = list(range(len(reftrace.trace)))
clean = p.line(xrange, reftrace.trace, line_color='blue')
glitch = p.line(xrange, glitchtrace.trace, line_color='red')
diff = p.line(xrange, abs(reftrace.trace - glitchtrace.trace) + 1, line_color='black')

legend = Legend(items=[
    LegendItem(label='glitch disabled', renderers=[clean]),
    LegendItem(label='glitch enabled', renderers=[glitch]),
    LegendItem(label='difference', renderers=[diff]),
])

p.add_layout(legend)
show(p)

Loading BokehJS ...

Zooming in around sample 4000 (because `scope.glitch.ext_offset * scope.clock.adc_mul = 4000`), we can indeed see that the glitch has a very obvious effect on the power trace.

It also, in our case, appears to have basically shifted the target's execution by a few cycles from that point onwards (but this could well vary from target to target, and even from run to run).

## But What Does the Glitch *Actually* Look Like?

You may simply be curious about what the glitch is actually doing to the target's power rail.

And that is absolutely an interesting thing to explore -- after all, it will *never* be a perfect square pulse from Vcc to 0 and back to Vcc.

For this you'll need an external oscilloscope. If you don't have one, we've included screen captures of what we observed on *our* setup, with Husky and a SAM4S target (again, remember that the glitch shape depends on the individual setup; it is also *highly* dependent on the target itself).

To trigger your oscilloscope, you have some options. You can of course trigger on TIO4. If you have a Husky, you may prefer to use the trigger/glitch output MCX port; this has the added advantage of giving you a copy of the digital signal that is actually driving the glitch MOSFET gate.

To see the "analog" glitch, we connect an oscilloscope probe to the SHUNTL pin of the CW313 or CW308 (using the HDR1 header on the former, or J16 on the latter).

On a single-board CW-lite, you can solder a header onto the middle pin of JP11, or either pin of JP6.

On a CW-Nano, the middle pin of the "GLITCH" header is what you want.

In [15]:
if scope._is_husky:
    scope.io.glitch_trig_mcx = 'glitch'

We'll also reduce the glitch duration to half of a target clock cycle:

In [16]:
scope.glitch.output = 'glitch_only'
scope.glitch.repeat = 1
scope.glitch.width = 4592//2

In [17]:
glitchtrace = cw.capture_trace(scope, target, bytearray(16), bytearray(16), as_int=False)

(ChipWhisperer Target WARNING|File SimpleSerial.py:454) Unexpected start to command: 


This is what we see on our scope:

![](../Images/glitch_hp.png)

In [18]:
scope.io.glitch_hp = True
scope.io.glitch_lp = True

glitchtrace = cw.capture_trace(scope, target, bytearray(16), bytearray(16), as_int=False)

(ChipWhisperer Target WARNING|File SimpleSerial.py:454) Unexpected start to command: 


If we turn on both MOSFETs (setting both `scope.io.glitch_hp` and `scope.io.glitch_hp` to `True`), we get quite a different picture:

![](../Images/glitch_hplp.png)

There is a huge parameter space to explore here:
- glitch duration
- `glitch_hp`, `glitch_lp`, or both *(N/A for CW-Nano which has a single glitch MOSFET)*
- multiple successive glitches of variable duration (Husky only)

You can get really fancy:

In [68]:
scope.glitch.num_glitches = 5
scope.glitch.repeat = [1, 2, 1, 1, 1]
scope.glitch.ext_offset = [1000, 20, 20, 0, 1]

scope.io.glitch_hp = False
scope.io.glitch_lp = True

glitchtrace = cw.capture_trace(scope, target, bytearray(16), bytearray(16), as_int=False)

(ChipWhisperer Target WARNING|File SimpleSerial.py:454) Unexpected start to command: 


![](../Images/glitch_fancy.png)

Explore, but don't forget that in the end, what the glitch looks like doesn't really matter; all that really matters is the effect that it has on the target.

## Other Ways to See Glitches

If what you care about is the shape of the digital pulse that's driving the glitch MOSFETs, there are a few other options:

### 1. Clock Glitching
The exact same hardware is responsible for generating the clock glitch shape and the voltage glitch shape; the only difference is that in the first case that internally generated glitch pulse is driven onto the output clock, whereas in the second case, it's driven to the glitch MOSFETs.

### 2. Logic Analyzer (Husky Only)
Husky's `scope.LA` logic analyzer can be used to capture the internal glitch pulse. [This demo notebook](https://github.com/newaetech/chipwhisperer-jupyter/blob/main/demos/husky/03%20-%20Husky%20Glitching.ipynb) teaches how. This is perhaps the easiest way to learn the use of all the `scope.glitch` glitch-shaping parameters.

### 3. Hardware Bitbanging (Husky Only)
One neat feature of Husky's `scope.bitbanger` module is that it lets you apply the internal glitch pulse onto the bit-banged data line. To learn more about this, check out the [hardware bitbanging demo notebook](https://github.com/newaetech/chipwhisperer-jupyter/blob/main/demos/husky/09%20-%20Husky%20Hardware%20Bitbanging.ipynb).


## But I Don't See the Glitches On the SHUNTL Pin!

If you can see the "digital" version of the glitch using one of the methods above (on the MCX port, clock glitch, `scope.LA`, or `scope.bitbanger`) but aren't seeing any evidence of the crowbar on the target supply, things to check include:
1. Did you connect an SMA cable from the capture hardware's "glitch" port to your target?
2. Is your SMA cable defective? (check for shorts)
3. (Shouldn't have to state this if you've read this far, but just in case:) Have you connected a target to your ChipWhisperer capture hardware?
4. Are you sure you're measuring the SHUNTL side and not the SHUNTH side?